# Fine-tune để "nhớ" luật, hay RAG?

Một yêu cầu rất hay gặp: "fine-tune model trên văn bản luật để nó trả lời luật". Module 13 mục 1 đã lập luận vì sao đó
thường là lựa chọn sai. Bài này **đo** điều đó trên Zalo Legal với một thước đo không cần LLM chấm: cho câu hỏi, model
phải trả đúng một trích dẫn dạng `<số hiệu văn bản>, Điều <số>`, và trích dẫn đúng khi trùng một điều trong qrels. Bạn
fine-tune Qwen2.5-0.5B theo hai cách (chỉ cặp câu hỏi, trích dẫn của split train; và thêm "đọc" tiêu đề của cả 61 425
điều), rồi so với BM25 của module 13 trên cùng 300 câu ghim. Đây cũng là bước đệm cho milestone fine-tune của capstone.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hằng, hàm. Điền vào chỗ `# TODO`.
- Cell `explore` cần kernel `ai-gpu` (GPU 4GB là đủ) và cài đặt BM25 của bạn ở module 13 bài Basic 02.
- Chấm: `uv run pytest tracks/ai-engineer/modules/14-fine-tuning-llm/exercises/advanced/02-finetune-vs-rag -q`

In [ ]:
import re

import numpy as np

## 1. Định dạng trích dẫn và thước đo

- `citation(uid) -> str`: uid của corpus (`"47/2011/tt-bca+7"`) thành `"47/2011/tt-bca, Điều 7"`.
- `parse_citation(text) -> str | None`: tìm (không phân biệt hoa thường) trích dẫn **đầu tiên** dạng
  `<số hiệu>, Điều <số>` trong `text`, trả uid chuẩn `"<số hiệu viết thường>+<số không có số 0 đầu>"`. Số hiệu bắt đầu
  bằng chữ hoặc số, gồm chữ, số, `.`, `/`, `-`; giữa số hiệu và chữ "Điều" là dấu phẩy (khoảng trắng tuỳ ý). Không có
  thì `None`.
- `citation_scores(preds, gold) -> dict`: `preds[i]` là uid (hoặc `None`), `gold[i]` là tập uid đúng của câu `i`. Trả
  `exact` (tỉ lệ câu có uid nằm trong tập đúng), `law` (tỉ lệ câu có **văn bản** đúng, phần trước dấu `+`) và `format`
  (tỉ lệ câu đọc được trích dẫn).

In [ ]:
CITE = re.compile(r"([0-9a-zđ][0-9a-zđ./\-]*)\s*,\s*điều\s+(\d+)", re.I)


def citation(uid: str) -> str:
    # TODO
    raise NotImplementedError


def parse_citation(text: str) -> str | None:
    # TODO
    raise NotImplementedError


def citation_scores(preds: list[str | None], gold: list[set[str]]) -> dict[str, float]:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: dữ liệu: corpus, câu hỏi, qrels train và 300 câu ghim của module 13
import json
import sys
import time
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))


def dataset_dir(name):
    try:
        from data.download_dataset import ensure
    except ImportError:
        path = ROOT / "data" / "raw" / name
        if not path.is_dir():
            raise RuntimeError(f"Chưa có dữ liệu: uv run python data/download_dataset.py {name}")
        return path
    return ensure(name)


ZALO = dataset_dir("zalo-legal")
corpus = {d["_id"]: d for d in map(json.loads, (ZALO / "corpus.jsonl").open(encoding="utf-8"))}
queries = {q["_id"]: q["text"] for q in map(json.loads, (ZALO / "queries.jsonl").open(encoding="utf-8"))}


def qrels(split):
    out = {}
    for r in map(json.loads, (ZALO / "qrels" / f"{split}.jsonl").open(encoding="utf-8")):
        out.setdefault(r["query-id"], set()).add(r["corpus-id"])
    return out


train_q, test_q = qrels("train"), qrels("test")
pinned, seen = list(test_q)[:300], list(train_q)[:300]
train_articles = {u for g in train_q.values() for u in g}
print(f"{len(train_q)} câu train, {len(train_articles)} điều được hỏi trong train; "
      f"câu ghim có điều đúng nằm trong số đó: {sum(bool(test_q[q] & train_articles) for q in pinned)}/300")

## 2. Chạy và viết báo cáo

Các cell còn lại là của bạn: BM25 top-1 (module 13) trên toàn corpus cho 300 câu ghim và 300 câu train đầu tiên;
model gốc chỉ bằng prompt; SFT LoRA trên cặp câu hỏi → trích dẫn của split train; SFT trên đó cộng một mẫu "tiêu đề
điều → trích dẫn" cho mọi điều của corpus. Mỗi cấu hình đo `citation_scores` trên cả hai tập câu, thời gian, VRAM.
Viết `REPORT.md` theo khung có sẵn, rồi làm phần trắc nghiệm của bài trong app.